# CCS Limpieza de Datos

En esta seccion se hara la limpieza de los dato ya explorados inicialmente, con el fin de limpiar nulos, eliminar duplicados y corregir tipos de datos

# CCS Data Cleaning

In this section, the previously explored data will be cleaned in order to remove nulls, eliminate duplicates, and correct data types.


In [28]:
# Import Libraries

import pandas as pd
import numpy as np

In [29]:
import psycopg2

# Configuración de conexión
conn = psycopg2.connect(
    host="localhost",
    port=5433,
    database="CCS",
    user="postgres",
    password="#74Hac63"
)

# Query SQL
query = '''SELECT
    cl."NUMERO_DE_IDENTIFICACION",
    cl."PRIMER_APELLIDO",
    cl."SEGUNDO_APELLIDO",
    cl."PRIMER_NOMBRE",
    cl."SEGUNDO_NOMBRE",
    cr."CURSO",
    fv."PRECIO_NETO",
    fv."FECHA_DE_PAGO",
    fv."RENOVACION",
    fv."FECHA_DE_VENTA",
    pf."PROFESION",
    gn."GENERO",
    cl."FECHA_DE_NACIMIENTO",
    ct."CIUDAD_REGION",
    md."MODALIDAD",
    pc."PROCEDENCIA",
    mp."MEDIO_DE_PAGO",
    rv."RESPONSABLE_VENTA",
    el."RESPONSABLE_VENTA" AS Elaboro

FROM fact_ventas AS fv
LEFT JOIN clientes as cl
    ON fv."CLIENTE_ID" = cl."CLIENTE_ID"
LEFT JOIN profesion as pf
    ON cl."PROFESION_ID" = pf."PROFESION_ID"
LEFT JOIN genero as gn
    ON gn."GENERO_ID" = cl."GENERO_ID"
LEFT JOIN ciudad_region as ct
    ON ct."CIUDAD_REGION_ID" = cl."CIUDAD_REGION_ID"
LEFT JOIN modalidad as md
    ON fv."MODALIDAD_ID" = md."MODALIDAD_ID"
LEFT JOIN procedencia as pc
    ON fv."PROCEDENCIA_ID" = pc."PROCEDENCIA_ID"
LEFT JOIN medio_de_pago as mp
    ON fv."MEDIO_DE_PAGO_ID" = mp."MEDIO_DE_PAGO_ID"
LEFT JOIN responsable_venta as rv
    ON rv."RESPONSABLE_VENTA_ID" = fv."RESPONSABLE_VENTA_ID"
LEFT JOIN responsable_venta as el
    ON el."RESPONSABLE_VENTA_ID" = fv."ELABORO"
LEFT JOIN cursos as cr
    ON fv."CURSO_ID" = cr."CURSO_ID";'''

# Ejecutar query y guardar en Excel
df = pd.read_sql_query(query, conn)
df.to_csv("./data_source/data_query_for_fac_ventas.csv", index=False)

conn.close()
print("Archivo guardado como resultado.xlsx")

Archivo guardado como resultado.xlsx


C:\Users\Hugo\AppData\Local\Temp\ipykernel_15672\2973967709.py:57: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


In [30]:
# Import data

df_fv = pd.read_csv('./data_source/data_query_for_fac_ventas.csv')

In [31]:
df_fv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7702 entries, 0 to 7701
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   NUMERO_DE_IDENTIFICACION  7702 non-null   int64  
 1   PRIMER_APELLIDO           7702 non-null   object 
 2   SEGUNDO_APELLIDO          7567 non-null   object 
 3   PRIMER_NOMBRE             7702 non-null   object 
 4   SEGUNDO_NOMBRE            7329 non-null   object 
 5   CURSO                     7701 non-null   object 
 6   PRECIO_NETO               7702 non-null   float64
 7   FECHA_DE_PAGO             7702 non-null   object 
 8   RENOVACION                7469 non-null   object 
 9   FECHA_DE_VENTA            7702 non-null   object 
 10  PROFESION                 5906 non-null   object 
 11  GENERO                    5803 non-null   object 
 12  FECHA_DE_NACIMIENTO       7699 non-null   object 
 13  CIUDAD_REGION             7528 non-null   object 
 14  MODALIDA

In [32]:
# Null values per column

df_fv.isnull().sum()

NUMERO_DE_IDENTIFICACION       0
PRIMER_APELLIDO                0
SEGUNDO_APELLIDO             135
PRIMER_NOMBRE                  0
SEGUNDO_NOMBRE               373
CURSO                          1
PRECIO_NETO                    0
FECHA_DE_PAGO                  0
RENOVACION                   233
FECHA_DE_VENTA                 0
PROFESION                   1796
GENERO                      1899
FECHA_DE_NACIMIENTO            3
CIUDAD_REGION                174
MODALIDAD                    523
PROCEDENCIA                  496
MEDIO_DE_PAGO                789
RESPONSABLE_VENTA             64
elaboro                      209
dtype: int64

## 1. Limpieza de datos nulos en columnas que no pueden tener datos nulos

Como se menciono en la seccion de entendimiento de los datos, existe ciertas columnas que no pueden tener datos nulos debido al funcionamientos del negocio,
"CURSO" es una de estas filas y se va a tratar como se indica a continuacion.

In [33]:
# Null percentage on COURSE regarding of total of COURSE registers
'''
Since the CURSOS column cannot contain null values, we evaluated which percentage of the total is represented by a single record.  
It is 0.01%, so we have decided to remove this row from the DataFrame.

'''

per_null = (df_fv["CURSO"].isnull().sum() / df_fv["CURSO"].value_counts().sum()) * 100
per_null

np.float64(0.01298532658096351)

In [34]:
# remove a single row from the DataFrame for null value on CURSOS

df_fv.dropna(subset=['CURSO'], inplace=True)

In [35]:
df_fv.shape

(7701, 19)

## 2. Manejo de datos nulos en columas que pueden tener datos nulos

En las columnas que si pueden tener datos nulos se va a dar un manejo para que no tenga como un dato nulo sino un valor estandar que sera "UNKNOWN" para datos categoricos y la media de los valores para datos numericos

In [36]:
# Action to fill each categorical column with a normaliced text "UNKNOWN"

col_list = [
 'SEGUNDO_APELLIDO',
 'SEGUNDO_NOMBRE',
 'RENOVACION',
 'PROFESION',
 'GENERO',
 'CIUDAD_REGION',
 'MODALIDAD',
 'PROCEDENCIA',
 'MEDIO_DE_PAGO',
 'RESPONSABLE_VENTA',
 'elaboro'
]

for col in col_list:
    df_fv.loc[df_fv[col].isnull(), col] = 'UNKNOWN'

In [37]:
# How mani null values column 'fecha de nacimiento' has?

# df_fv[df_fv['FECHA_DE_NACIMIENTO'].isna()]

In [38]:
# Action to change default dates '1900-01-01' by NaT
# Tis is not a representative value for the general analysis so will leave as NaT
values = ["1900-01-01"]

df_fv.loc[df_fv['FECHA_DE_NACIMIENTO'].isin(values), 'FECHA_DE_NACIMIENTO'] = np.nan

In [39]:
df_fv.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7701 entries, 0 to 7701
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   NUMERO_DE_IDENTIFICACION  7701 non-null   int64  
 1   PRIMER_APELLIDO           7701 non-null   object 
 2   SEGUNDO_APELLIDO          7701 non-null   object 
 3   PRIMER_NOMBRE             7701 non-null   object 
 4   SEGUNDO_NOMBRE            7701 non-null   object 
 5   CURSO                     7701 non-null   object 
 6   PRECIO_NETO               7701 non-null   float64
 7   FECHA_DE_PAGO             7701 non-null   object 
 8   RENOVACION                7701 non-null   object 
 9   FECHA_DE_VENTA            7701 non-null   object 
 10  PROFESION                 7701 non-null   object 
 11  GENERO                    7701 non-null   object 
 12  FECHA_DE_NACIMIENTO       5234 non-null   object 
 13  CIUDAD_REGION             7701 non-null   object 
 14  MODALIDAD    

In [40]:
# Now we will normalice column names to uppercase

df_fv.columns = df_fv.columns.str.upper()

In [41]:
#df_fv.sample(2)

## 3. Correccion de el tipo de dato

En este paso vamos a corregir el tipo de dato asignado a cada variable de DataFrame

In [42]:
data_type = {
    'numeric' : ['PRECIO_NETO'],
    'category' : [
        'NUMERO_DE_IDENTIFICACION',
        'CURSO',
        'RENOVACION',
        'PROFESION',
        'GENERO',
        'CIUDAD_REGION',
        'MODALIDAD',
        'PROCEDENCIA',
        'MEDIO_DE_PAGO',
        'RESPONSABLE_VENTA'
        ],
    'date' : [
        'FECHA_DE_PAGO',
        'FECHA_DE_VENTA',
        'FECHA_DE_NACIMIENTO'
        ]
}


In [43]:
def change_dtype(df, key, value):
    if key == 'numeric':
        for col in value:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            print("Numeric type changed!")
    elif key == 'category':
        for col in value:
            df[col] = df[col].astype('category')
            print("Category type changed!")
    elif key == 'date':
        for col in value:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            print("Date type changed!")
    else:
        print("ERROR!")

In [44]:
for key, value in data_type.items():
    change_dtype(df_fv, key, value)

Numeric type changed!
Category type changed!
Category type changed!
Category type changed!
Category type changed!
Category type changed!
Category type changed!
Category type changed!
Category type changed!
Category type changed!
Category type changed!
Date type changed!
Date type changed!
Date type changed!


In [45]:
df_fv.dtypes

NUMERO_DE_IDENTIFICACION          category
PRIMER_APELLIDO                     object
SEGUNDO_APELLIDO                    object
PRIMER_NOMBRE                       object
SEGUNDO_NOMBRE                      object
CURSO                             category
PRECIO_NETO                        float64
FECHA_DE_PAGO               datetime64[ns]
RENOVACION                        category
FECHA_DE_VENTA              datetime64[ns]
PROFESION                         category
GENERO                            category
FECHA_DE_NACIMIENTO         datetime64[ns]
CIUDAD_REGION                     category
MODALIDAD                         category
PROCEDENCIA                       category
MEDIO_DE_PAGO                     category
RESPONSABLE_VENTA                 category
ELABORO                             object
dtype: object

In [46]:
df_fv.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7701 entries, 0 to 7701
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   NUMERO_DE_IDENTIFICACION  7701 non-null   category      
 1   PRIMER_APELLIDO           7701 non-null   object        
 2   SEGUNDO_APELLIDO          7701 non-null   object        
 3   PRIMER_NOMBRE             7701 non-null   object        
 4   SEGUNDO_NOMBRE            7701 non-null   object        
 5   CURSO                     7701 non-null   category      
 6   PRECIO_NETO               7701 non-null   float64       
 7   FECHA_DE_PAGO             7701 non-null   datetime64[ns]
 8   RENOVACION                7701 non-null   category      
 9   FECHA_DE_VENTA            7701 non-null   datetime64[ns]
 10  PROFESION                 7701 non-null   category      
 11  GENERO                    7701 non-null   category      
 12  FECHA_DE_NACIMIENTO      

## 4. Creacion de columnas derivadas

In [47]:
# Age columns

today = pd.Timestamp.today()

df_fv['EDAD'] = (today.year - df_fv['FECHA_DE_NACIMIENTO'].dt.year)

In [48]:
# Year of Sale

df_fv['ANIO_VENTA'] = df_fv['FECHA_DE_PAGO'].dt.year

In [49]:
# Month of Sale

df_fv['MES_VENTA'] = df_fv['FECHA_DE_PAGO'].dt.month_name(locale="es_ES")

In [50]:
# Define Cleint Ranges
'''
MENOR -> 0 - 18
JOVEN -> 18 - 30
ADULTO -> 30 - 50
SENIOR -> 50 -120
'''

bins = [0, 18, 30, 50, 120]
labels = ["Menor", "Joven", "Adulto", "Senior"]

df_fv['GRUPO_EDAD'] = pd.cut(df_fv['EDAD'], bins= bins, labels=labels)

In [51]:
#df_fv.sample(5)

In [52]:
df_fv['GRUPO_EDAD'].value_counts()


GRUPO_EDAD
Adulto    2953
Joven     1816
Senior     405
Menor       28
Name: count, dtype: int64

In [53]:
df_fv.columns.to_list()

['NUMERO_DE_IDENTIFICACION',
 'PRIMER_APELLIDO',
 'SEGUNDO_APELLIDO',
 'PRIMER_NOMBRE',
 'SEGUNDO_NOMBRE',
 'CURSO',
 'PRECIO_NETO',
 'FECHA_DE_PAGO',
 'RENOVACION',
 'FECHA_DE_VENTA',
 'PROFESION',
 'GENERO',
 'FECHA_DE_NACIMIENTO',
 'CIUDAD_REGION',
 'MODALIDAD',
 'PROCEDENCIA',
 'MEDIO_DE_PAGO',
 'RESPONSABLE_VENTA',
 'ELABORO',
 'EDAD',
 'ANIO_VENTA',
 'MES_VENTA',
 'GRUPO_EDAD']

In [54]:
df_fv.to_excel('./data_source/data_query_for_fac_ventas.xlsx', index=False)